# 17b · T4 (fp16) vs L4 (bf16) parity, before Phase C may run on a T4

**Used only on Dhanush's word** (25 Nov), and only if no L4 is available for Phase C. Phase C runs on a T4 only if this check **passes**; any miss means Phase C waits for an L4.

**Two steps:**
1. **Freeze the sample (CPU runtime, about 0.01 CU):** set `FREEZE_ONLY = True` and Run all. It draws 300 edited + 300 real validation images (seed 42, salt `t4_parity_v1`) from notebook 08's L4 per-image files and freezes them, by SHA-256, in `eval_subsets.json` (key `T4_parity`). The hash goes into the spec card **before** the run.
2. **The parity run (T4, about 0.35 CU at 1.28 CU/h, estimate):** set `FREEZE_ONLY = False` and `CU_BALANCE_START`, then Run all. It stops unless the frozen sample is intact.

**Model:** unified seed 42, final configuration (`eviground_ctx_unified_seed42/best.pt`), fp16 autocast on the T4. The tiling (full coverage) and the pass (`runner.diagnosis_pass`) are the same as notebooks 08 and 09.

**Pre-registered pass rules (all three):**
1. **P2 dev, 486 images** (frozen CP4 dev split, from the verified `p2` tar): pooled edited IoU within **±0.005** of the L4 value **0.5101** (CP4 G2, 18 Nov).
2. **300 real validation images:** verdict agreement with notebook 08's L4 verdicts **≥ 99.5%**.
3. **300 edited validation images:** median |T4 − L4| per-image IoU **≤ 0.01**.

On the L4, P(edited) was never saved, and notebook 08's edited file has no predicted label, so verdicts are compared on real images only.

**Result** (send to Claude): `Paper 1/runs/checkpoints/t4_parity_seed42.json` and `Paper 1/runs/cp4/compute_t4_parity.json`, with the balance.

In [ ]:
# Cell 1: mount Drive, get the code, install it
import os, subprocess, sys

DRY = os.environ.get('EVIGROUND_DRY_RUN') == '1'   # CI / local test: fake sources, temp folders
REPO_URL = 'github.com/DHANUSHKUMMARI/Paper1.git'
BRANCH = 'main'                                    # 'claude/nice-curie-3tzxzr' before the merge

if DRY:
    REPO = os.path.abspath('..' if os.path.basename(os.getcwd()) == 'notebooks' else '.')
else:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    REPO = '/content/Paper1'
    if not os.path.exists(REPO):
        try:
            tok = userdata.get('GITHUB_TOKEN')       # never printed
        except Exception:
            tok = None
        url = f'https://{tok}@{REPO_URL}' if tok else f'https://{REPO_URL}'
        r = subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, url, REPO],
                           capture_output=True, text=True)
        if r.returncode != 0:                        # stderr is not shown: it can contain the URL
            raise RuntimeError('git clone failed: check GITHUB_TOKEN (Secrets panel) and BRANCH')
        subprocess.run(['git', '-C', REPO, 'remote', 'set-url', 'origin', f'https://{REPO_URL}'],
                       check=True)
    else:
        subprocess.run(['git', '-C', REPO, 'pull', '--ff-only'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', REPO + '[train]'], check=True)
    # Colab ships torchao 0.10; peft refuses a torchao older than 0.16 with an ImportError when it
    # adds the LoRA layers, although EviGround never uses torchao: remove an old one.
    import importlib, importlib.metadata as _md
    from packaging.version import Version
    try:
        if Version(_md.version('torchao')) < Version('0.16.0'):
            subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'], check=True)
            importlib.invalidate_caches()
    except _md.PackageNotFoundError:
        pass

sys.path.insert(0, os.path.join(REPO, 'src'))
# Re-running in a runtime that already imported eviground would keep the OLD code in memory
# although git pulled the new one: drop it so the next import reads the files on disk.
for _name in [m for m in sys.modules if m == 'eviground' or m.startswith('eviground.')]:
    del sys.modules[_name]
print('repo:', REPO, '| branch:', BRANCH, '| dry run:', DRY)

In [ ]:
# Cell 2: settings
FREEZE_ONLY = True          # step 1 (CPU): draw and freeze the 300 + 300 sample; False: the T4 parity run
CU_BALANCE_START = None     # REQUIRED for the T4 run: the balance Colab shows now
SEED = 42
RUN_EXPERIMENT = 'eviground_ctx'   # the final unified model (CP4 fix)
DRY_DIR = 'eviground_t4parity_dry'
PROFILES = ['unified']
FIGURES_ONLY = False

import csv, datetime, json, logging, shutil, tempfile, time
from pathlib import Path
logging.basicConfig(level=logging.INFO, force=True)
import numpy as np, torch
from eviground.config import load_config
from eviground.data import pack, testcache
from eviground.env import get_paths, session_info
from eviground.eval import devsplit, parity, runner, subsets, testsets
from eviground.eval.tiled import per_image
from eviground.utils import session

T0 = time.time()
paths = get_paths()
if DRY:
    from eviground.env import Paths
    _dry = Path(tempfile.gettempdir()) / DRY_DIR
    shutil.rmtree(_dry, ignore_errors=True)
    paths = Paths(root=_dry / 'drive' / 'Paper 1', local_data=_dry / 'data')
    FREEZE_ONLY = False      # the dry run does both steps
paths.ensure()
GPU = session_info().get('gpu_name')
GPU_TYPE = session.gpu_type(GPU)
CU_PER_HOUR = session.CU_RATES.get(GPU_TYPE, session.CU_RATES['cpu'])
info = session_info(cu_per_hour=CU_PER_HOUR)
CONFIG = 'debug' if DRY else 'base'
HF_CACHE = paths.cache / 'hf' if DRY else Path('/content/hf_cache')
OVERRIDES = [f'model.semantic.cache_dir={HF_CACHE}']
cfg = load_config(CONFIG, OVERRIDES)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP = runner.set_amp_dtype('fp16' if GPU_TYPE == 'T4' else 'bf16')
EXPERIMENT = cfg.get('experiment', 'eviground')
CK = paths.runs / 'checkpoints'
CP1_MANIFEST = CK / 'cp1' / 'manifest_cp1.csv.gz'
OUT = paths.runs / 'cp4'
OUT.mkdir(parents=True, exist_ok=True)
SUBSETS_FILE = CK / 'eval_subsets.json'
DEV_SPLIT = CK / 'cp4_dev_split.json'
GATE = CK / f'cp4_generalization_gate_seed{SEED}_ctx.json'
REF_EDITED, REF_REAL = OUT / f'diagnosis_edited_seed{SEED}_ctx.csv', OUT / f'diagnosis_real_seed{SEED}_ctx.csv'
CACHE = paths.runs / 'cache' / 'tests'
RESULT = CK / f't4_parity_seed{SEED}.json'
def run_dir(profile):
    exp = RUN_EXPERIMENT if profile == 'unified' else EXPERIMENT
    return paths.runs / f'{exp}_{profile}_seed{SEED}'
print('commit', info['git_commit'], '| gpu', GPU, f'({GPU_TYPE})', '| autocast', AMP, '| freeze only', FREEZE_ONLY)

In [ ]:
# Cell 2b: pre-flight
EST_CU = 0.0 if DRY else (0.01 if FREEZE_ONLY else 0.35)
NEEDED = {'L4 edited per-image (notebook 08)': REF_EDITED, 'L4 real per-image (notebook 08)': REF_REAL}
if not FREEZE_ONLY:
    NEEDED.update({'unified checkpoint': run_dir('unified') / 'best.pt', 'CP4 G2 result (L4)': GATE,
                   'CP4 dev split': DEV_SPLIT, 'eval_subsets.json': SUBSETS_FILE,
                   'tar p2': testcache.paths(CACHE, 'p2')[0]})
if DRY:
    NEEDED = {}   # the dry run builds its inputs below
else:
    if not FREEZE_ONLY:
        assert GPU_TYPE == 'T4', f'the parity run is for a T4 (fp16), found {GPU}'
        assert isinstance(CU_BALANCE_START, (int, float)), 'type the Colab balance into CU_BALANCE_START (cell 2)'
PREFLIGHT = session.preflight(
    {'notebook': '17b T4 parity', 'step': 'freeze the sample' if FREEZE_ONLY else 'parity run',
     'gpu': GPU, 'autocast': AMP, 'result file': RESULT},
    paths=NEEDED, est_cu=EST_CU, cu_left=CU_BALANCE_START)

In [ ]:
# Cell 3: shards and the model (not in the freeze-only step)
if not FREEZE_ONLY:

    assert not FIGURES_ONLY, 'FIGURES_ONLY = True: run only cells 1, 2 and the last two cells (figures, release)'
    SHARDS = paths.shards
    CP1_RESULT = None
    if DRY:   # fake world -> CP1 -> shards -> tiny trained models (one epoch each)
        from eviground.data.cp1 import CP1Config, run_cp1
        from eviground.data.download import ensure_data
        from eviground.data.fake_cp1 import cp1_overrides, make_cp1_world
        from eviground.train.trainer import Trainer
        sources = make_cp1_world(paths.datasets, paths.root.parent / 'fake_remote', coco_images=40)
        ensure_data('cp1', paths=paths, sources=sources, stage_local=True, overrides=cp1_overrides(40))
        CP1_RESULT = run_cp1(paths, CP1Config(workers=2, crops_per_class=60, blockiness_per_group=20,
                                              base_unclear_max=0.2, diffseg_card_splits={'train': 18, 'validation': 12}),
                             out_dir=CK / 'cp1', cache_dir=paths.cache / 'cp1', progress=None)
        pack.pack(CP1_MANIFEST, SHARDS, free_bytes=10**12, margin=0,
                  shard_bytes=200_000, sample_per_source=5, progress=None)
        for profile in PROFILES:
            Trainer(load_config(CONFIG, OVERRIDES + ['train.epochs=1']), SHARDS, run_dir(profile),
                    seed=SEED, profile=profile, resume=None, log=None).fit()
    LOCAL_SHARDS = paths.local_data / 'shards'
    print(pack.stage_shards(SHARDS, LOCAL_SHARDS, progress=print))
    SHARDS = LOCAL_SHARDS
    BEST = run_dir('unified') / 'best.pt'
    model = runner.load_trained(cfg.model, BEST, DEVICE)
    predict = runner.joint_predictor(model, None, DEVICE)   # the model head, as in notebooks 08 and 09
    INDEX = {r['uid']: r for r in pack.read_index(SHARDS / 'index.parquet')}
    print('loaded', BEST, '| shard rows', len(INDEX))

In [ ]:
# Cell 4: (dry run only) the L4 references, written by the same code paths as notebooks 08 and 09
if DRY:
    _val = [r for r in INDEX.values() if r['split'] == 'val' and r['label'] == 'real']
    _val += [r for r in INDEX.values() if r['split'] == 'train' and r['label'] == 'edited']   # as notebook 08's dry run
    _ref = runner.diagnosis_pass(predict, testsets.shard_items(SHARDS, _val), DEVICE, ceilings=False, sweep=False)
    with open(REF_EDITED, 'w', newline='') as fh:
        w = csv.writer(fh); w.writerow(['uid', 'model_iou'])
        for r in _ref['edited']:
            w.writerow([r['uid'], per_image(r['heads']['model'])['iou']])
    with open(REF_REAL, 'w', newline='') as fh:
        w = csv.writer(fh); w.writerow(['uid', 'model_label'])
        for r in _ref['real']:
            w.writerow([r['uid'], r['heads']['model']['label']])
    _edited_train = [r for r in INDEX.values() if r['split'] == 'train' and r['label'] == 'edited']
    _p2 = runner.diagnosis_pass(predict, testsets.shard_items(SHARDS, _edited_train), DEVICE, ceilings=False, sweep=False)
    GATE.write_text(json.dumps({'p2': {'model': {'all': runner.p2_summary(_p2['edited'], 'model')['all']}}}))
    print('dry references:', len(_ref['edited']), 'edited,', len(_ref['real']), 'real')

In [ ]:
# Cell 5: the 300 + 300 sample from notebook 08's L4 files, frozen in eval_subsets.json (key T4_parity)
REF_E = {r['uid']: float(r['model_iou']) for r in parity.read_csv(REF_EDITED) if r['model_iou'] not in ('', 'None')}
REF_R = {r['uid']: r['model_label'] for r in parity.read_csv(REF_REAL)}
SAMPLE = parity.sample_parity(REF_E, REF_R, n=parity.N_EACH, seed=SEED)
STORED = json.loads(SUBSETS_FILE.read_text()) if SUBSETS_FILE.is_file() else {}
if 'T4_parity' in STORED:
    assert parity.intact(STORED['T4_parity']), 'T4_parity in eval_subsets.json was changed: stop and ask Dhanush'
    assert STORED['T4_parity']['sha256'] == SAMPLE['sha256'], 'the frozen T4_parity sample differs from a fresh draw: stop and ask'
    print('T4_parity already frozen')
else:
    assert FREEZE_ONLY or DRY, 'no frozen T4_parity sample: run step 1 (FREEZE_ONLY = True) and lay the spec card first'
    STORED['T4_parity'] = SAMPLE
    SUBSETS_FILE.parent.mkdir(parents=True, exist_ok=True)
    SUBSETS_FILE.write_text(json.dumps(STORED, indent=1))
    print('T4_parity frozen in', SUBSETS_FILE)
print(f"edited {SAMPLE['edited']['images']} of {SAMPLE['edited']['pool']} | real {SAMPLE['real']['images']} of "
      f"{SAMPLE['real']['pool']} | sha256 {SAMPLE['sha256']}")

In [ ]:
# Cell 6: the T4 passes (P2 dev from the p2 tar; the sampled validation images from the shards)
T4 = None
if not FREEZE_ONLY:
    if DRY:
        p2_items = testsets.shard_items(SHARDS, [r for r in INDEX.values() if r['split'] == 'train' and r['label'] == 'edited'])
    else:
        dev = set(devsplit.load(DEV_SPLIT)['dev_uids'])
        p2_rows = [r for r in testcache.stage(CACHE, 'p2', paths.local_data / 'parity') if r['uid'] in dev]
        assert len(p2_rows) == 486, f'P2 dev: {len(p2_rows)} rows, expected 486'
        p2_items = testsets.file_items(p2_rows)
    P2 = runner.diagnosis_pass(predict, p2_items, DEVICE, ceilings=False, sweep=False)
    P2_ALL = runner.p2_summary(P2['edited'], 'model')['all']
    rows = [INDEX[u] for u in SAMPLE['edited']['uids'] + SAMPLE['real']['uids']]
    VAL = runner.diagnosis_pass(predict, testsets.shard_items(SHARDS, rows), DEVICE, ceilings=False, sweep=False)
    T4 = {'p2_iou': P2_ALL['iou_edited'],
          'edited_iou': {r['uid']: per_image(r['heads']['model'])['iou'] for r in VAL['edited']},
          'real_label': {r['uid']: r['heads']['model']['label'] for r in VAL['real']},
          'stats': {'p2': P2['stats'], 'val': VAL['stats']}}
    print('P2 pooled edited IoU', round(T4['p2_iou'], 6), '|', P2['stats'], '|', VAL['stats'])

In [ ]:
# Cell 7: the verdict, the result file, the compute record, then release
if T4 is not None:
    REF_P2 = json.loads(GATE.read_text())['p2']['model']['all']['iou_edited']
    if not DRY:
        assert REF_P2 == parity.L4_P2_IOU, (REF_P2, parity.L4_P2_IOU)
    VERDICT = parity.parity_verdict(
        T4['p2_iou'], T4['edited_iou'], T4['real_label'],
        {u: REF_E[u] for u in SAMPLE['edited']['uids']}, {u: REF_R[u] for u in SAMPLE['real']['uids']},
        ref_p2_iou=REF_P2)
    l4_seconds, l4_tiles = 104.5, 4130   # the L4 P2 pass of CP4 G2 (18 Nov)
    t4 = T4['stats']['p2']
    RESULT_DATA = {
        'step': 'T4 (fp16) vs L4 (bf16) parity', 'dry_run': DRY, 'branch': BRANCH, 'seed': SEED,
        'session': info, 'gpu': GPU, 'autocast': AMP, 'sample': {k: v for k, v in SAMPLE.items() if k not in ('edited', 'real')}
        | {'edited_sha256': SAMPLE['edited']['sha256'], 'real_sha256': SAMPLE['real']['sha256']},
        'verdict': VERDICT, 'pass': VERDICT['pass'], 'stats': T4['stats'],
        'p2_seconds_per_tile': {'t4': round(t4['seconds'] / max(t4['tiles'], 1), 5), 'l4': round(l4_seconds / l4_tiles, 5)},
        'edited_per_image': {u: {'t4': T4['edited_iou'][u], 'l4': REF_E[u]} for u in SAMPLE['edited']['uids']},
        'utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
    }
    RESULT.write_text(json.dumps(RESULT_DATA, indent=1, default=str))
    print('PARITY', 'PASS' if VERDICT['pass'] else 'FAIL', {k: v['pass'] for k, v in VERDICT.items() if k != 'pass'})
    print(json.dumps({k: {kk: vv for kk, vv in v.items() if kk != 'disagreeing_uids'} for k, v in VERDICT.items() if k != 'pass'}, indent=1))
COMPUTE = session.finish(OUT, T0, gpu=GPU or 'cpu', name='compute_t4_parity.json', balance_start=CU_BALANCE_START,
                         extra={'git_commit': info['git_commit'], 'est_cu': EST_CU, 'freeze_only': FREEZE_ONLY})
if DRY:
    assert VERDICT['pass'], VERDICT                 # the same code on the same device agrees with itself
    _bad = parity.parity_verdict(T4['p2_iou'], {u: x + 0.02 for u, x in T4['edited_iou'].items()}, T4['real_label'],
                                 {u: REF_E[u] for u in SAMPLE['edited']['uids']}, {u: REF_R[u] for u in SAMPLE['real']['uids']},
                                 ref_p2_iou=REF_P2)
    assert not _bad['pass']                         # a shifted result fails
    assert parity.intact(json.loads(SUBSETS_FILE.read_text())['T4_parity'])
RELEASED = session.release(DRY)
if DRY:
    assert RELEASED == 'dry', RELEASED